# GNSS Filtering Results

Plot precomputed plasma delays, tracked GNSS satellites, Monte Carlo trajectory errors, and the corresponding 3-sigma covariance envelopes generated by `pixi run run-gnss-pipeline`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"figure.figsize": (11, 7), "axes.grid": True})

output_candidates = [Path("output/gnss_filtering"), Path("../../output/gnss_filtering")]
output_dir = next((p for p in output_candidates if p.exists()), output_candidates[0])
trajectory_files = sorted(output_dir.glob("trajectory_mc*.csv"))
summary_path = output_dir / "summary.csv"
required_component_columns = {
    "r_error_m",
    "t_error_m",
    "n_error_m",
    "rdot_error_mps",
    "tdot_error_mps",
    "ndot_error_mps",
    "r_3sigma_m",
    "t_3sigma_m",
    "n_3sigma_m",
    "rdot_3sigma_mps",
    "tdot_3sigma_mps",
    "ndot_3sigma_mps",
    "clock_bias_3sigma_m",
    "clock_drift_3sigma_mps",
    "num_tracked_satellites",
}

if not trajectory_files:
    raise FileNotFoundError(f"No trajectory files found under {output_dir.resolve()}")

summary = pd.read_csv(summary_path) if summary_path.exists() else None
all_runs = [(path, pd.read_csv(path)) for path in trajectory_files]
runs = [df for _, df in all_runs if required_component_columns.issubset(df.columns)]
legacy_files = [
    path.name
    for path, df in all_runs
    if not required_component_columns.issubset(df.columns)
]

if not runs:
    raise ValueError(
        "No trajectory files contain the component-error/3-sigma columns. "
        "Re-run `pixi run run-gnss-pipeline --skip-delays` to regenerate outputs with the new schema."
    )

len(runs), [
    path.name for path, _ in all_runs if path.name not in legacy_files
], legacy_files

In [ ]:
if summary is not None:
    display(summary)

if legacy_files:
    print(
        f"Ignoring legacy trajectory files without component covariance columns: {legacy_files}"
    )

for df in runs:
    df["time_min"] = df["t"] / 60.0

In [ ]:
delay_path = output_dir / "precomputed_delays.csv"
if not delay_path.exists():
    delay_path = output_dir / "precomputed_links.csv"

if delay_path.exists():
    delays = pd.read_csv(delay_path)
    delays["time_min"] = (delays["t_tdb"] - delays["t_tdb"].min()) / 60.0
    fig, ax = plt.subplots(figsize=(11, 5))
    for (const, prn, freq), group in delays.groupby(["gnss_const", "prn", "frequency"]):
        ax.plot(
            group["time_min"],
            group["ionosphere_plasma_delay_m"],
            linewidth=1.0,
            label=f"{const}-{int(prn):02d} {freq}",
        )
    ax.set_xlabel("Receiver app elapsed coordinate time [min]")
    ax.set_ylabel("delay [m]")
    ax.set_title("Precomputed Plasma/Ionosphere Delays")
    ax.legend(loc="best", ncols=2)
    fig.tight_layout()
else:
    print("No precomputed link or delay file found.")

In [ ]:
def plot_component_group(columns, sigma_columns, ylabel, title):
    fig, axes = plt.subplots(
        len(columns), 1, sharex=True, figsize=(11, 2.6 * len(columns))
    )
    if len(columns) == 1:
        axes = [axes]
    for ax, col, sig_col in zip(axes, columns, sigma_columns):
        for df in runs:
            t = df["time_min"].to_numpy()
            err = df[col].to_numpy()
            sig = df[sig_col].to_numpy()
            ax.plot(
                t, err, linewidth=1.0, alpha=0.8, label=f"MC {int(df['mc'].iloc[0])}"
            )
            ax.fill_between(t, -sig, sig, alpha=0.10)
        ax.set_ylabel(ylabel)
        ax.set_title(col)
    axes[-1].set_xlabel("Receiver app elapsed coordinate time [min]")
    axes[0].legend(loc="best", ncols=2)
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    return fig


plot_component_group(
    ["r_error_m", "t_error_m", "n_error_m"],
    ["r_3sigma_m", "t_3sigma_m", "n_3sigma_m"],
    "error [m]",
    "RTN Position Errors with 3-Sigma Bounds",
);

In [ ]:
plot_component_group(
    ["rdot_error_mps", "tdot_error_mps", "ndot_error_mps"],
    ["rdot_3sigma_mps", "tdot_3sigma_mps", "ndot_3sigma_mps"],
    "error [m/s]",
    "RTN Velocity Errors with 3-Sigma Bounds",
);

In [ ]:
plot_component_group(
    ["clock_bias_error_m", "clock_drift_error_mps"],
    ["clock_bias_3sigma_m", "clock_drift_3sigma_mps"],
    "error",
    "Clock Errors with 3-Sigma Bounds",
);

In [ ]:
srp_columns = {"srp_coeff_error_m2_kg", "srp_coeff_3sigma_m2_kg"}
if srp_columns.issubset(runs[0].columns) and any(
    df["srp_coeff_error_m2_kg"].notna().any() for df in runs
):
    plot_component_group(
        ["srp_coeff_error_m2_kg"],
        ["srp_coeff_3sigma_m2_kg"],
        "error [m^2/kg]",
        "SRP Coefficient Error with 3-Sigma Bounds",
    )
else:
    print("SRP coefficient estimation is disabled for these runs.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for df in runs:
    ax.step(
        df["time_min"],
        df["num_tracked_satellites"],
        where="post",
        label=f"MC {int(df['mc'].iloc[0])}",
    )
ax.set_xlabel("Receiver app elapsed coordinate time [min]")
ax.set_ylabel("tracked satellites")
ax.set_title("Tracked Satellites")
ax.legend(loc="best", ncols=2)
fig.tight_layout();

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for df in runs:
    ax.plot(df["time_min"], df["pos_error_m"], label=f"MC {int(df['mc'].iloc[0])}")
ax.set_xlabel("Receiver app elapsed coordinate time [min]")
ax.set_ylabel("position error norm [m]")
ax.set_title("Position Error Norm")
ax.legend(loc="best", ncols=2)
fig.tight_layout();